# Engines

> bioMONAI training engines


In [ ]:
#| default_exp engines

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export

# =================================
# Scientific / data
# =================================
import numpy as np
import pandas as pd

# =================================
# Visualization
# =================================
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =================================
# Imaging
# =================================
# from skimage import util

# =================================
# PyTorch
# =================================
import torch.optim as toptim
from torch.cuda import is_available as is_cuda_available
from torch.nn.init import kaiming_normal_

# =================================
# fastai
# =================================
# import fastai.losses
# import fastai.metrics
# import fastai.optimizer

from fastai.callback.core import Callback
from fastai.callback.all import *

from fastai.data.all import (
    DataLoaders, Path, trainable_params, delegates,
    hasattrs, List, L, Normalize
)

from fastai.optimizer import Adam, OptimWrapper, Optimizer

from fastai.vision.all import (
    Any, BypassNewMeta, CSVLogger, ClassificationInterpretation,
    DataBlock, DisplayedTransform, Learner, ShowGraphCallback,
    create_vision_model, create_timm_model, default_split,
    get_c, ifnone, minimum, model_meta, slide, steep, store_attr, valley
)

# =================================
# fastcore
# =================================
from fastcore.script import risinstance

# =================================
# bioMONAI
# =================================
from bioMONAI.utils import *
from bioMONAI.datasets import download_medmnist

The engine module provides advanced functionalities for model training, including configurable training loops and evaluation functions tailored for bioinformatics applications. This module is significantly valuable when there is a need for specific workflows and pipelines that meet specific requirements. For this reason, the classes fastTrainer and visionTrainer have been created, providing tailored implementations inheriting from the Learner class. 


## fastai Engine

FastTrainer is used for training models in bioinformatics applications, where specific loss functions and optimizers oriented to biological data can be used. 

In [ ]:
#| export
class fastTrainer(Learner):
    """
    A custom implementation of the FastAI Learner class for training models in bioinformatics applications.

    """
    
    def __init__(self, 
                 dataloaders: DataLoaders, # The DataLoader objects containing training and validation datasets.
                 model: callable, # A callable model that will be trained on the dataset.
                 loss_fn: Any | None = None, # The loss function to optimize during training. If None, defaults to a suitable default.
                 optimizer: Optimizer | OptimWrapper = Adam, # The optimizer function to use. Defaults to Adam if not specified.
                 lr: float | slice = 1e-3, # Learning rate for the optimizer. Can be a float or a slice object for learning rate scheduling.
                 splitter: callable = trainable_params, # 
                 callbacks: Callback | MutableSequence | None = None, # A callable that determines which parameters of the model should be updated during training.
                 metrics: Any | MutableSequence | None = None, # Optional list of callback functions to customize training behavior.
                 csv_log: bool = False, # Metrics to evaluate the performance of the model during training.
                 show_graph: bool = True, # Whether to log training history to a CSV file. If True, logs will be appended to 'history.csv'.
                 show_summary: bool = False, # The base directory where models are saved or loaded from. Defaults to None.
                 find_lr: bool = False, # Subdirectory within the base path where trained models are stored. Default is 'models'.
                 find_lr_fn = valley, # Weight decay factor for optimization. Defaults to None.
                 path: str | Path | None = None, # Whether to apply weight decay to batch normalization and bias parameters.
                 model_dir: str | Path = 'models', # Whether to update the batch normalization statistics during training.
                 wd: float | int | None = None, 
                 wd_bn_bias: bool = False, 
                 train_bn: bool = True, 
                 moms: tuple = (0.95,0.85,0.95), # Tuple of tuples representing the momentum values for different layers in the model. Defaults to FastAI's default settings if not specified.
                 default_cbs: bool = True, # Automatically include default callbacks such as ShowGraphCallback and CSVLogger.
                 ):
        cbs = callbacks if callbacks is not None else []  # Ensure cbs is a list
        if default_cbs:
            if show_graph:
                cbs.append(ShowGraphCallback())
            if csv_log:
                cbs.append(CSVLogger(fname='history.csv', append=False))
        
        super().__init__(dataloaders, model, loss_fn, optimizer, lr, splitter, cbs, metrics, path, model_dir, wd, wd_bn_bias, train_bn, moms)
        
        if show_summary:
                print(self.summary())
        if find_lr:
                lr = self.lr_find(suggest_funcs=find_lr_fn)
                self.lr = float('%.1g'%(lr))
                print('Inferred learning rate: ', self.lr)
        

    @classmethod
    def from_yaml(cls, dataloaders, model, yaml_path):
        """
        Method to read from a YAML file and obtain the parameters for FastTrainer.
        """
        # Read the configuration from the yaml file and replace None strings with Nonetype values
        config = read_yaml(yaml_path)
        config = {key: (None if value == "None" else value) for key, value in config.items()}

        
        
        # OBTAIN THE LOSS FUNCTION
        loss_str = config.get('loss_fn', None)
        # Look for the loss function within the variables and check if its valid
        if loss_str == None:
            loss_func = None
        else:
            lf_cls = globals().get(loss_str) or getattr(fm, loss_str, None)
            if lf_cls is None:
                raise ValueError(f"Loss function '{loss_str}' not found or invalid.")
            loss_func = lf_cls()


        # OBTAIN THE OPTIMIZER
        opt_str = config.get('optimizer', 'Adam')
        # Look for the optimizer within the variables and check if its valid
        if isinstance(opt_str, str):
            opt_cls = globals().get(opt_str, None)
            if opt_cls is None and hasattr(fastai.optimizer, opt_str):
                opt_cls = getattr(fastai.optimizer, opt_str)
            if opt_cls is None and hasattr(toptim, opt_str):
                opt_cls = getattr(toptim, opt_str)
            if opt_cls is None or not callable(opt_cls):
                raise ValueError(f"Optimizer '{opt_str}' not found or invalid.")
            opt_func = opt_cls


        # OBTAIN THE METRICS
        metrics_cfg = config.get('metrics', [])
        # Look for the metrics within the variables and check if they are valid
        valid_metrics = []
        if metrics_cfg == None:
            valid_metrics = None
        else:
            valid_metrics = dictlist_to_funclist(metrics_cfg)

        # OBTAIN THE CALLBACKS
        callbacks = []
        cbs = config.get('callbacks', [])
        callbacks = dictlist_to_funclist(cbs)

        # OBTAIN THE SPLITTER
        splitter_str = config.get('splitter', trainable_params)
        if isinstance(splitter_str, str):
            splitter_func = globals().get(splitter_str) or getattr(fastai.learner, splitter_str, None)
        elif splitter_str is None:
            splitter_func = trainable_params 
        else:
                splitter_func = splitter_str


        # OBTAIN OTHER PARAMETERS FROM THE YAML CONFIGURATION 
        lr = config.get('lr', 1e-3)
        csv_log = config.get('csv_log', False)
        show_graph = config.get('show_graph', True)
        show_summary = config.get('show_summary', False)
        path = config.get('path', None)
        model_dir = config.get('model_dir', 'models')
        wd = config.get('wd', None)
        wd_bn_bias = config.get('wd_bn_bias', False)
        train_bn = config.get('train_bn', True)
        moms = config.get('moms', (0.95,0.85,0.95))
   


        # Return all the parameters
        return cls(dataloaders = dataloaders, model = model, loss_fn = loss_func, optimizer = opt_func,
            lr = lr, splitter = splitter_func, callbacks = callbacks, metrics = valid_metrics, path = path,
            model_dir = model_dir, wd = wd, wd_bn_bias = wd_bn_bias, train_bn = train_bn, moms = moms,
            csv_log = csv_log, show_graph = show_graph, show_summary = show_summary          
        )        

#### Example: train a model with configuration from a YAML file.

In [ ]:
from monai.networks.nets import SEResNet50
from bioMONAI.data import BioDataLoaders
from bioMONAI.metrics import BalancedAccuracy, Precision, accuracy

In [ ]:
# Import the data
image_path = '_data'
info = download_medmnist('bloodmnist', image_path, download_only=True)
batch_size = 32
path = Path(image_path)/'bloodmnist'
path_train = path/'train'
path_val = path/'val'

Dataset 'bloodmnist' is already downloaded and available in '_data/bloodmnist'.


In [ ]:
# Define the dataloader
data = BioDataLoaders.class_from_folder(
    path,
    train='train',
    valid='val',
    vocab=info['label'],
    batch_tfms=None,
    bs=batch_size)

# Define the model
model = SEResNet50(spatial_dims=2,
                   in_channels=3,   
                   num_classes=8)    

In [ ]:
# Define the trainer with configuration from a YAML file 
# yaml_path = "./data_examples/sample_config.yml"
# trainer = fastTrainer.from_yaml(data, model, yaml_path)

# # Train the model
# trainer.fit(1)


In [ ]:
# print(trainer.recorder.metric_names)

In [ ]:
#| export
def _add_norm(dls, meta, pretrained, n_in=3):
    if not pretrained: return
    stats = meta.get('stats')
    if stats is None: return
    if n_in != len(stats[0]): return
    if not dls.after_batch.fs.filter(risinstance(Normalize)):
        dls.add_tfms([Normalize.from_stats(*stats)],'after_batch')

def _timm_norm(dls, cfg, pretrained, n_in=3):
    if not pretrained: return
    if n_in != len(cfg['mean']): return
    if not dls.after_batch.fs.filter(risinstance(Normalize)):
        tfm = Normalize.from_stats(cfg['mean'],cfg['std'])
        dls.add_tfms([tfm],'after_batch')

VisionTrainer is used for computer vision applications, where image normalization or other computer vision related settings are needed.

In [ ]:
#| export
@delegates(create_vision_model)
def visionTrainer(  dataloaders: DataLoaders, # The DataLoader objects containing training and validation datasets.
                    model: callable, # A callable model that will be trained on the dataset.
                    normalize=True, 
                    n_out=None, 
                    pretrained=True, 
                    weights=None,
                    # Trainer args
                    loss_fn: Any | None = None, # The loss function to optimize during training. If None, defaults to a suitable default.
                    optimizer: Optimizer | OptimWrapper = Adam, # The optimizer function to use. Defaults to Adam if not specified.
                    lr: float | slice = 1e-3, # Learning rate for the optimizer. Can be a float or a slice object for learning rate scheduling.
                    splitter: callable = trainable_params, # 
                    callbacks: Callback | MutableSequence | None = None, # A callable that determines which parameters of the model should be updated during training.
                    metrics: Any | MutableSequence | None = None, # Optional list of callback functions to customize training behavior.
                    csv_log: bool = False, # Metrics to evaluate the performance of the model during training.
                    show_graph: bool = True, # Whether to log training history to a CSV file. If True, logs will be appended to 'history.csv'.
                    show_summary: bool = False, # The base directory where models are saved or loaded from. Defaults to None.
                    find_lr: bool = False, # Subdirectory within the base path where trained models are stored. Default is 'models'.
                    find_lr_fn = valley, # Weight decay factor for optimization. Defaults to None.
                    path: str | Path | None = None, # Whether to apply weight decay to batch normalization and bias parameters.
                    model_dir: str | Path = 'models', # Whether to update the batch normalization statistics during training.
                    wd: float | int | None = None, 
                    wd_bn_bias: bool = False, 
                    train_bn: bool = True, 
                    moms: tuple = (0.95,0.85,0.95), # Tuple of tuples representing the momentum values for different layers in the model. Defaults to FastAI's default settings if not specified.
                    default_cbs: bool = True, # Automatically include default callbacks such as ShowGraphCallback and CSVLogger.
                    # model & head args
                    cut=None, 
                    init=kaiming_normal_, 
                    custom_head=None, 
                    concat_pool=True, 
                    pool=True,
                    lin_ftrs=None, 
                    ps=0.5, 
                    first_bn=True, 
                    bn_final=False, 
                    lin_first=False, 
                    y_range=None, 
                    **kwargs):
    "Build a vision trainer from `dataloaders` and `model`"
    if n_out is None: n_out = get_c(dataloaders)
    assert n_out, "`n_out` is not defined, and could not be inferred from data, set `dataloaders.c` or pass `n_out`"
    meta = model_meta.get(model, {'cut':cut, 'split':default_split})
    model_args = dict(init=init, custom_head=custom_head, concat_pool=concat_pool, pool=pool, lin_ftrs=lin_ftrs, ps=ps,
                      first_bn=first_bn, bn_final=bn_final, lin_first=lin_first, y_range=y_range, **kwargs)
    n_in = kwargs['n_in'] if 'n_in' in kwargs else 3
    if isinstance(model, str):
        model,cfg = create_timm_model(model, n_out, default_split, pretrained, **model_args)
        if normalize: _timm_norm(dataloaders, cfg, pretrained, n_in)
    else:
        if normalize: _add_norm(dataloaders, meta, pretrained, n_in)
        model = create_vision_model(model, n_out, pretrained=pretrained, weights=weights, **model_args)

    splitter = ifnone(splitter, meta['split'])
    trainer = fastTrainer(dataloaders, model, loss_fn=loss_fn, optimizer=optimizer, lr=lr, splitter=splitter, callbacks=callbacks, csv_log=csv_log, 
                        show_graph=show_graph, show_summary=show_summary, find_lr=find_lr, find_lr_fn=find_lr_fn, metrics=metrics, path=path, 
                        model_dir=model_dir, wd=wd, wd_bn_bias=wd_bn_bias, train_bn=train_bn, moms=moms, default_cbs=default_cbs)
    if pretrained: trainer.freeze()
    # keep track of args for loggers
    store_attr('model,normalize,n_out,pretrained', self=trainer, **kwargs)
    return trainer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()